# JupyterHub sanity check - train a garment classifier with MindSpore

This notebook is used to confirm the correct operation of my Kubernetes-based JupyterHub notebook execution environment with the following specifications.

| Resource type | Specification |
| --- | --- |
| vCPU | `4` |
| Memory | `16Gi` |
| NPU | 1x Ascend 310P |
| OS | Ubuntu 24.04 LTS |
| Python version | `3.12.11` |
| CANN version | `8.5.0` |
| MindSpore version | `2.10.0` |

In [1]:
!cat /sys/fs/cgroup/cpu.max

400000 100000


In [2]:
!cat /sys/fs/cgroup/memory.max

17179869184


In [3]:
!npu-smi info

+--------------------------------------------------------------------------------------------------------+
| npu-smi v1.0                                     Version: 24.1.rc4.b999                                |
+-------------------------------+-----------------+------------------------------------------------------+
| NPU     Name                  | Health          | Power(W)     Temp(C)           Hugepages-Usage(page) |
| Chip    Device                | Bus-Id          | AICore(%)    Memory-Usage(MB)                        |
+===============================+=================+======================================================+
| 1792    310P1                 | OK              | NA           52                0     / 0             |
| 0       0                     | 0000:08:00.0    | 0            1785 / 89608                            |
+===============================+=================+======================================================+
+-------------------------------+----

In [4]:
!cat /etc/os-release

PRETTY_NAME="Ubuntu 24.04.3 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.3 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo


In [5]:
!python --version

Python 3.12.11


In [6]:
!cat /usr/local/Ascend/cann/opp/version.info

Version=8.5.0
version_dir=cann
required_package_runtime_version=">=8.5.0"
required_package_ge-executor_version=">=8.5.0"
required_package_metadef_version=">=8.5.0"
required_package_asc-devkit_version=">=8.5.0"
required_package_bisheng-compiler_version=">=8.5.0"


In [7]:
%pip show mindspore

Name: mindspore
Version: 2.10.0
Summary: MindSpore is a new open source deep learning training/inference framework that could be used for mobile, edge and cloud scenarios.
Home-page: https://www.mindspore.cn
Author: The MindSpore Authors
Author-email: contact@mindspore.cn
License: Apache 2.0
Location: /opt/conda/lib/python3.12/site-packages
Requires: asttokens, astunparse, dill, numpy, packaging, pillow, protobuf, psutil, safetensors, scipy
Required-by: 
Note: you may need to restart the kernel to use updated packages.


The example below is adapted from [`00-garment-classifier-mindspore.py`](https://github.com/DonaldKellett/my-ascend-python/blob/a11c879efaccf35be7909a398ee0048d46539909/orangepiaipro-20t/04-upgrade-cann-mindspore/00-garment-classifier-mindspore.py) which in turn is based on the [Training with PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html) official tutorial.

In [8]:
%%time
import gzip
import mindspore
import mindspore.amp as amp
import mindspore.dataset as ds
import mindspore.dataset.transforms as transforms
import mindspore.dataset.vision as vision
import mindspore.nn as nn
import mindspore.ops as ops
import mlflow
import os
import urllib.request

from mindspore import dtype as mstype
from mindspore.train import Callback, LossMonitor, Model

def transform_ds(dataset, batch_size):
    image_transforms = [
        vision.Resize(size=(28, 28)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW(),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=batch_size, drop_remainder=False)
    return dataset

"""
MLflow logging callback with accuracy
"""
class MLflowLogging(Callback):
    def __init__(self, run_name, network_type, loss_fn, learning_rate, batch_size, epochs, weight_decay=0.0, momentum=0.0, optimizer='sgd'):
        super().__init__()
        self.run_name = run_name
        self.network_type = network_type
        self.loss_fn = loss_fn
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.weight_decay = weight_decay
        self.momentum = momentum
        self.optimizer = optimizer

        self.run = mlflow.start_run(run_name=self.run_name)
        hyperparameters = {
            'learning_rate': self.learning_rate,
            'weight_decay': self.weight_decay,
            'momentum': self.momentum,
            'loss_fn': self.loss_fn,
            'optimizer': self.optimizer,
            'batch_size': self.batch_size,
            'network_type': self.network_type,
            'epochs': self.epochs
        }
        mlflow.log_params(hyperparameters)

    def on_train_step_end(self, run_context):
        cb_params = run_context.original_args()
        current_loss = cb_params.net_outputs.asnumpy().mean()
        mlflow.log_metric('train_loss', current_loss, step=cb_params.cur_step_num)

    def on_train_epoch_end(self, run_context):
        cb_params = run_context.original_args()
        if hasattr(cb_params, 'eval_results') and cb_params.eval_results:
            val_loss = cb_params.eval_results.get('loss', 0.0)
            val_accuracy = cb_params.eval_results.get('accuracy', 0.0)
            mlflow.log_metric('val_loss', val_loss, step=cb_params.cur_epoch_num)
            mlflow.log_metric('val_accuracy', val_accuracy, step=cb_params.cur_epoch_num)

    def on_train_end(self, run_context):
        mlflow.end_run()

def main():
    mindspore.set_device(device_target='Ascend', device_id=0)
    
    MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
    print(f'Using MLflow tracking URI: {MLFLOW_TRACKING_URI}')
    
    experiment_name = '00-jupyterhub-sanity-check'
    experiment = mlflow.set_experiment(experiment_name=experiment_name)

    dataset_dir = 'data/fashion/'
    os.makedirs(dataset_dir, exist_ok=True)

    prefix_url = 'https://assets.donaldsebleung.com/datasets/fashion-mnist'
    X_train_url = f'{prefix_url}/train-images-idx3-ubyte.gz'
    y_train_url = f'{prefix_url}/train-labels-idx1-ubyte.gz'
    X_test_url = f'{prefix_url}/t10k-images-idx3-ubyte.gz'
    y_test_url = f'{prefix_url}/t10k-labels-idx1-ubyte.gz'

    X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
    y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
    X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
    y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

    with urllib.request.urlopen(X_train_url) as response:
        with open(X_train_path, 'wb') as out_file:
            data_gzip = response.read()
            data = gzip.decompress(data_gzip)
            out_file.write(data)

    with urllib.request.urlopen(y_train_url) as response:
        with open(y_train_path, 'wb') as out_file:
            data_gzip = response.read()
            data = gzip.decompress(data_gzip)
            out_file.write(data)

    with urllib.request.urlopen(X_test_url) as response:
        with open(X_test_path, 'wb') as out_file:
            data_gzip = response.read()
            data = gzip.decompress(data_gzip)
            out_file.write(data)

    with urllib.request.urlopen(y_test_url) as response:
        with open(y_test_path, 'wb') as out_file:
            data_gzip = response.read()
            data = gzip.decompress(data_gzip)
            out_file.write(data)

    train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
    test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

    batch_size = 128
    train_ds = transform_ds(dataset=train_ds, batch_size=batch_size)
    test_ds = transform_ds(dataset=test_ds, batch_size=batch_size)

    net = nn.SequentialCell([
        nn.Conv2d(1, 6, kernel_size=5, pad_mode='valid'),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Conv2d(6, 16, kernel_size=5, pad_mode='valid'),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Flatten(),
        nn.Dense(256, 120, activation='relu'),
        nn.Dense(120, 84, activation='relu'),
        nn.Dense(84, 10)
    ])

    learning_rate = 0.1
    net_amp = amp.auto_mixed_precision(network=net, amp_level='O2')
    loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
    optimizer = nn.SGD(params=net_amp.trainable_params(), learning_rate=learning_rate)

    run_name = '00-garment-classifier-mindspore'
    network_type = 'lenet'
    loss_fn_str = 'softmax_cross_entropy'
    epochs = 10
    model = Model(
        network=net_amp,
        loss_fn=loss_fn,
        optimizer=optimizer,
        metrics={'accuracy', 'loss'}
    )
    callbacks = [
        LossMonitor(per_print_times=10),
        MLflowLogging(
            run_name=run_name,
            network_type=network_type,
            loss_fn=loss_fn_str,
            learning_rate=learning_rate,
            batch_size=batch_size,
            epochs=epochs
        )
    ]
    model.fit(
        epoch=epochs,
        train_dataset=train_ds,
        valid_dataset=test_ds,
        callbacks=callbacks,
        dataset_sink_mode=False
    )

if __name__ == '__main__':
    main()

Using MLflow tracking URI: http://component-mlflow-mlflow.mlflow:5000


/usr/local/Ascend/cann-8.5.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:97: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)
/usr/local/Ascend/cann-8.5.0/opp/built-in/op_impl/ai_core/tbe/impl/ops_legacy/dynamic/gelu_grad_v2.py:157: SyntaxWarning: invalid escape sequence '\h'
  gelu_grad_erf = erfc(-\hat{x}) / 2 + (1 /sqrt(Pi)) * (\hat{x}) * exp(-\hat{x}^2)


epoch: 1 step: 10, loss is 2.2984671592712402
epoch: 1 step: 20, loss is 2.3013720512390137
epoch: 1 step: 30, loss is 2.3042593002319336
epoch: 1 step: 40, loss is 2.293682813644409
epoch: 1 step: 50, loss is 2.2992911338806152
epoch: 1 step: 60, loss is 2.299511194229126
epoch: 1 step: 70, loss is 2.293757438659668
epoch: 1 step: 80, loss is 2.288008213043213
epoch: 1 step: 90, loss is 2.2907896041870117
epoch: 1 step: 100, loss is 2.2839815616607666
epoch: 1 step: 110, loss is 2.278200626373291
epoch: 1 step: 120, loss is 2.2753210067749023
epoch: 1 step: 130, loss is 2.233999729156494
epoch: 1 step: 140, loss is 2.176598072052002
epoch: 1 step: 150, loss is 1.7568283081054688
epoch: 1 step: 160, loss is 2.1909022331237793
epoch: 1 step: 170, loss is 2.3392157554626465
epoch: 1 step: 180, loss is 1.3608825206756592
epoch: 1 step: 190, loss is 1.3480825424194336
epoch: 1 step: 200, loss is 1.1852155923843384
epoch: 1 step: 210, loss is 1.0676214694976807
epoch: 1 step: 220, loss is 1